# Stock Price Prediction - LSTM Training (Colab Version)

This notebook is designed to train LSTM models for Indian stocks.

### Instructions:
1. Upload your `indian_stocks_all_history.csv` to the 'content' folder in Colab.
2. Run all cells to train models for selected tickers.
3. The last cell will automatically zip and download the models for you.

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import os
from google.colab import files
import shutil

In [4]:
import os
import shutil

# !pip install kagglehub

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("krishchaudhary14/indian-stock-market-full-5-year-history")

print("Path to dataset files:", path)

100%|██████████| 80.8M/80.8M [00:04<00:00, 17.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/krishchaudhary14/indian-stock-market-full-5-year-history/versions/3


In [6]:
# The downloaded 'path' is a directory. We need to find the CSV file inside it.
# Assuming the CSV is directly inside the downloaded folder and named 'indian_stocks_all_history.csv'

source_csv_path = os.path.join(path, 'indian_stocks_all_history.csv')
destination_csv_path = 'indian_stocks_all_history.csv'

# Check if the source file exists before copying
if os.path.exists(source_csv_path):
    shutil.copy(source_csv_path, destination_csv_path)
    print(f"Successfully copied '{source_csv_path}' to '{destination_csv_path}'")
else:
    print(f"Error: '{source_csv_path}' not found. Please check the dataset structure.")

# Verify that the file is in the current directory
print("Files in current directory:", os.listdir('.'))

Successfully copied '/root/.cache/kagglehub/datasets/krishchaudhary14/indian-stock-market-full-5-year-history/versions/3/indian_stocks_all_history.csv' to 'indian_stocks_all_history.csv'
Files in current directory: ['.config', 'indian_stocks_all_history.csv', 'sample_data']


## 1. Data Processor Class

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

class DataProcessor:
    def __init__(self, file_path='indian_stocks_all_history.csv'):
        self.file_path = file_path
        self.scaler = MinMaxScaler(feature_range=(0, 1))

    def load_data(self, ticker):
        cols = ['ticker', 'date', 'open', 'high', 'low', 'close', 'volume']
        # Use low_memory=False to avoid warnings on large files
        df = pd.read_csv(self.file_path, usecols=cols, low_memory=False)
        df = df[df['ticker'] == ticker].copy()
        df['date'] = pd.to_datetime(df['date'])
        df.sort_values('date', inplace=True)
        df.set_index('date', inplace=True)
        return df

    def add_features(self, df):
        df['SMA_20'] = df['close'].rolling(window=20).mean()
        df['SMA_50'] = df['close'].rolling(window=50).mean()
        df['Returns'] = df['close'].pct_change()
        df['Volatility'] = df['Returns'].rolling(window=20).std()
        df.dropna(inplace=True)
        return df

    def prepare_lstm_data(self, df, feature_col='close', window_size=60):
        data = df[[feature_col]].values
        scaled_data = self.scaler.fit_transform(data)
        X, y = [], []
        for i in range(window_size, len(scaled_data)):
            X.append(scaled_data[i-window_size:i, 0])
            y.append(scaled_data[i, 0])
        X, y = np.array(X), np.array(y)
        X = np.reshape(X, (X.shape[0], X.shape[1], 1))
        return X, y, scaled_data

## 2. Model Trainer Class

In [11]:
class ModelTrainer:
    def __init__(self, model_path='stock_lstm.h5'):
        self.model_path = model_path
        if not os.path.exists('models'):
            os.makedirs('models')

    def build_model(self, input_shape):
        model = Sequential([
            LSTM(units=50, return_sequences=True, input_shape=input_shape),
            Dropout(0.2),
            LSTM(units=50, return_sequences=False),
            Dropout(0.2),
            Dense(units=25),
            Dense(units=1)
        ])
        model.compile(optimizer='adam', loss='mean_squared_error')
        return model

    def train(self, X_train, y_train, epochs=10, batch_size=32):
        input_shape = (X_train.shape[1], 1)
        model = self.build_model(input_shape)
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=1)
        model.save(self.model_path)
        return model

## 3. Training Loop

In [12]:
tickers = ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK']
processor = DataProcessor()

for ticker in tickers:
    print(f"\n--- Training for {ticker} ---")
    try:
        df = processor.load_data(ticker)
        df = processor.add_features(df)
        X, y, _ = processor.prepare_lstm_data(df)

        trainer = ModelTrainer(model_path=f'models/{ticker}_lstm.h5')
        trainer.train(X, y, epochs=10)
        print(f"Finished {ticker}")
    except Exception as e:
        print(f"Error training {ticker}: {e}")


--- Training for RELIANCE ---
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


74/74 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0216
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0063
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0058
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0058
Epoch 5/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0050
Epoch 6/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0047
Epoch 7/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0048
Epoch 8/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0045
Epoch 9/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0042
Epoch 10/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0041


Finished RELIANCE

--- Training for TCS ---
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

74/74 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.0165
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0040
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0035
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0031
Epoch 5/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0029
Epoch 6/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0026
Epoch 7/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0025
Epoch 8/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0023
Epoch 9/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0021
Epoch 10/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0022


Finished INFY

--- Training for ICICIBANK ---
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


74/74 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0237
Epoch 2/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0038
Epoch 3/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0030
Epoch 4/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0026
Epoch 5/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0024
Epoch 6/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0023
Epoch 7/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0019
Epoch 8/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0020
Epoch 9/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0016
Epoch 10/10
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0016


Finished ICICIBANK


## 4. Zip and Download Models

In [13]:
import shutil
from google.colab import files

print("Zipping models...")
shutil.make_archive('trained_models', 'zip', 'models')
print("Downloading zip file...")
files.download('trained_models.zip')

Zipping models...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>